In [1]:
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import KFold, GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import matplotlib.pyplot as plt
plt.style.use('seaborn-darkgrid')
import seaborn as sns
sns.set_theme(style='darkgrid')
from sklearn.preprocessing import StandardScaler, PowerTransformer
state_space=3504

drive='main_path'

In [2]:
## bonus, saving the nss_district_code19 and spei_district_code (map codes) to a separate file
mapfile=pd.read_csv(f"{drive}/xgboost_shap/feature_engineering/regdf_v2Mar25.csv")
mapfile=mapfile[['nss_district_code19','spei_district_code']].drop_duplicates().reset_index(drop=True)
print(mapfile.shape)
mapfile.to_csv(f"{drive}/nss_to_spei_code.csv",index=False)

(657, 2)


# Making the model data from the survey data and climate data

In [3]:
alldf=pd.read_csv("C://Users/nb/Desktop/paper_2/xgboost_shap/feature_engineering/regdf_v2Mar25.csv")
alldf['cost_energy']=alldf[["cost_diesel", "cost_electricity"]].sum(axis=1)
alldf['cost_labour']=alldf[['cost_labour_animal', 'cost_labour_human']].sum(axis=1)
alldf['cost_finance']=alldf[['cost_crop_insurance','cost_land_lease','cost_interest_on_crop_loan']].sum(axis=1)
alldf["cost_pest"]=alldf[["cost_chemical_pesticides","cost_bio_pesticides"]].sum(axis=1)
alldf['cost_machine']=alldf[["cost_minor_machine_repairs", "cost_crop_machinery"]].sum(axis=1)
alldf['farm_crop_yield']=alldf["farm_output_kg"]/alldf["farm_land_m2"]
alldf["cost_fert"]=alldf[["cost_chemical_fertilizer","cost_bio_fertilizer","cost_manure"]].sum(axis=1)
alldf=alldf.rename(columns={'household_cons_exp_monthly':'hh_cons_exp_pm'})
state_dummies = pd.get_dummies(alldf['state'].astype(str), prefix='state',drop_first=True)
alldf=pd.concat([alldf,state_dummies],axis=1)
states=[i for i in alldf.columns if 'state_' in i]
dropcols=["cost_diesel","cost_electricity","cost_labour_animal","cost_labour_human","cost_crop_insurance",
          "cost_land_lease","cost_interest_on_crop_loan","cost_bio_fertilizer","cost_manure","farm_output_kg",
          "cost_minor_machine_repairs","cost_crop_machinery",'share_soiltype0', 'share_soiltype1', 'share_soiltype2',
          'share_soiltype3', 'share_soiltype4', 'share_soiltype5','share_soiltype6', 'share_soiltype7',
         'HHID_dt', 'crop_code','ln_farm_crop_yield', 'state']
alldf=alldf.drop(dropcols,axis=1)
ricedf=alldf[(alldf['crop_name']=='paddy')&(alldf['visit_no']==1)].drop(['crop_name','visit_no'],axis=1)
wheatdf=alldf[(alldf['crop_name']=='wheat')&(alldf['visit_no']==2)].drop(['crop_name','visit_no'],axis=1)

In [5]:
ricedf['tp_mean'].mean()

0.0073280531215767055

In [7]:
ricedf.shape

(22332, 84)

In [6]:
ricedf=ricedf[ricedf['farm_crop_yield']<=500] ## one observation is weird
print('Rice shape = ',ricedf.shape)
print('Wheat shape = ',wheatdf.shape)
ricedf.to_csv(f"{drive}/mainricedf.csv",index=False)
wheatdf.to_csv(f"{drive}/mainwheatdf.csv",index=False)

Rice shape =  (22331, 84)
Wheat shape =  (13595, 84)


In [7]:
ricedf=pd.read_csv(f"{drive}/mainricedf.csv")
wheatdf=pd.read_csv(f"{drive}/mainwheatdf.csv")

In [8]:
yes_no=['is_non_backward_caste','if_farmer_organisation',
        'if_kisan_credit_card', 'if_soil_health_card',
        'share_own_and_possessed', 'if_land_irrigated',
        'primary_irrigation_canal',
        'primary_irrigation_gw', 'primary_irrigation_mixed',
        'primary_irrigation_msw', 'primary_irrigation_others']
cont=['cost_energy','cost_labour', 'cost_finance', 'cost_fert',
             'cost_pest', 'cost_machine', 'cost_irrigation',
             'cost_seeds','hh_cons_exp_pm']
contpm2=[f'{i}_pm2' for i in cont]
climate=['median_speiSPEI-4MO','max_speiSPEI-4MO', 'min_speiSPEI-4MO', 'std_speiSPEI-4MO', 
         'tp_mean','tp_std', 't2m_mean', 't2m_max', 't2m_min', 't2m_std', 
         'slhf_mean','slhf_max','slhf_min', 'slhf_std', 
         'ssr_mean', 'ssr_max', 'ssr_min','ssr_std']
states=[ 'state_10', 'state_11', 'state_12',
       'state_13', 'state_14', 'state_15', 'state_16', 'state_17', 'state_18',
       'state_19', 'state_2', 'state_20', 'state_21', 'state_22', 'state_23',
       'state_24', 'state_25', 'state_26', 'state_27', 'state_28', 'state_29',
       'state_3', 'state_30', 'state_32', 'state_33', 'state_34', 'state_36',
       'state_4', 'state_5', 'state_6', 'state_7', 'state_8', 'state_9'] # taken from alldf.columns

In [10]:
for ft in cont:
    ricedf[f'{ft}_pm2']=ricedf[ft]/ricedf['farm_land_m2']
    wheatdf[f'{ft}_pm2']=wheatdf[ft]/wheatdf['farm_land_m2']

## Separating in-sample and out-of-sample, with descriptive statistics

There are two stratifications performed here: 
1. Allowing the models to learn within-districts - Can the model **interpolate**? - This allows policy analysis and review at the ground-level where climate and district features are known - so that the uncertainty is purely at the farm-level.
2. Not allowing the models to learn within-districts - In-Sample and Out-of-Sample districts are mutually exclusive so that the model is expected to **extrapolate**. Here the generalization is truly multi-level because uncertainty is at the farm-level and at the district level.

### District-Size Split 
Districts are ranked by their size and in-sample and out-of-sample are selected so that districts of different sizes are included in both. This ensures that neither sample space is dominated by particularly large districts. First the districts are ranked by size and 80% of all observations in each size-bin is set to in-sample and 20% is set to out-of-sample.

### For Rice

In [11]:
ricedf['district_size']=ricedf[['nss_district_code19','farm_crop_yield']].copy().groupby('nss_district_code19').transform('count')
print(ricedf.shape)
ricedf=ricedf[ricedf['district_size']>=5] ## restricting data to districts with at least a few observations
ricedf['district_size_ranks']=pd.qcut(ricedf['district_size'].copy(),duplicates='drop',labels=range(10),q=10)

isrice,oosrice=train_test_split(ricedf,test_size=0.2,stratify=ricedf['district_size_ranks'],random_state=state_space)

(22331, 94)


In [6]:
isrice=pd.read_csv(f"{drive}/xgboost_shap/feature_engineering/shared_insample_rice.csv")
print('Number of districts in-sample rice: ',isrice.spei_district_code.nunique())
oosrice=pd.read_csv(f"{drive}/xgboost_shap/feature_engineering/shared_oosample_rice.csv")
print('OOS Rice',oosrice.spei_district_code.nunique())

Number of districts in-sample rice:  491
OOS Rice 474


In [9]:
print('*** In-Sample rice data descriptive statistics ***')
print()
print('                   **** Farm-Level Metrics ****')
for ft in yes_no:
    print(f'The percentage of {ft} = {isrice[ft].mean()*100:.2f}')
print()
print(np.round(isrice[['farm_crop_yield','farm_land_m2']+contpm2].describe(),2))
print()
print('                   **** District Climate Metrics ****')
print(np.round(isrice[climate].describe(),4))

*** In-Sample rice data descriptive statistics ***

                   **** Farm-Level Metrics ****
The percentage of is_non_backward_caste = 26.13
The percentage of if_farmer_organisation = 4.16
The percentage of if_kisan_credit_card = 18.12
The percentage of if_soil_health_card = 1.22
The percentage of share_own_and_possessed = 82.57
The percentage of if_land_irrigated = 63.63
The percentage of primary_irrigation_canal = 13.16
The percentage of primary_irrigation_gw = 45.29
The percentage of primary_irrigation_mixed = 0.97
The percentage of primary_irrigation_msw = 3.02
The percentage of primary_irrigation_others = 2.27

       farm_crop_yield  farm_land_m2  cost_energy_pm2  cost_labour_pm2  \
count         17756.00      17756.00         17756.00         17756.00   
mean              0.33       7969.44             0.12             1.33   
std               0.18      10531.25             0.37             1.79   
min               0.00         80.94             0.00             0.00   

In [10]:
print('*** Out-of-Sample rice data descriptive statistics ***')
print()
print('                   **** Farm-Level Metrics ****')
for ft in yes_no:
    print(f'The percentage of {ft} = {oosrice[ft].mean()*100:.2f}')
print()
print(np.round(oosrice[['farm_crop_yield','farm_land_m2']+contpm2].describe(),2))
print()
print('                   **** District Climate Metrics ****')
print(np.round(oosrice[climate].describe(),4))

*** Out-of-Sample rice data descriptive statistics ***

                   **** Farm-Level Metrics ****
The percentage of is_non_backward_caste = 26.26
The percentage of if_farmer_organisation = 4.23
The percentage of if_kisan_credit_card = 17.36
The percentage of if_soil_health_card = 1.40
The percentage of share_own_and_possessed = 81.48
The percentage of if_land_irrigated = 63.54
The percentage of primary_irrigation_canal = 13.72
The percentage of primary_irrigation_gw = 44.46
The percentage of primary_irrigation_mixed = 0.95
The percentage of primary_irrigation_msw = 3.13
The percentage of primary_irrigation_others = 2.25

       farm_crop_yield  farm_land_m2  cost_energy_pm2  cost_labour_pm2  \
count          4440.00       4440.00          4440.00          4440.00   
mean              0.33       8037.57             0.12             1.32   
std               0.18       9675.63             0.31             1.62   
min               0.00        121.41             0.00             0.0

In [14]:
print('Saving shared-district Rice Data')
isrice.to_csv(f"{drive}/xgboost_shap/feature_engineering/shared_insample_rice.csv",index=False)
oosrice.to_csv(f"{drive}/xgboost_shap/feature_engineering/shared_oosample_rice.csv",index=False)
print('Rice data saved ~~')

Saving shared-district Rice Data
Rice data saved ~~


### For Wheat

In [45]:
wheatdf['district_size']=wheatdf[['nss_district_code19','farm_crop_yield']].copy().groupby('nss_district_code19').transform('count')
print(wheatdf.shape)
wheatdf=wheatdf[wheatdf['district_size']>=5] ## restricting data to districts with at least a few observations
wheatdf['district_size_ranks']=pd.qcut(wheatdf['district_size'].copy(),duplicates='drop',labels=range(10),q=10)

iswheat,ooswheat=train_test_split(wheatdf,test_size=0.2,stratify=wheatdf['district_size_ranks'],random_state=state_space)

(13518, 95)


In [11]:
iswheat=pd.read_csv(f"{drive}/xgboost_shap/feature_engineering/shared_insample_wheat.csv")
print('Number of districts in-sample wheat: ',iswheat.spei_district_code.nunique())
ooswheat=pd.read_csv(f"{drive}/xgboost_shap/feature_engineering/shared_oosample_wheat.csv")
print('OOS wheat',ooswheat.spei_district_code.nunique())

Number of districts in-sample wheat:  323
OOS wheat 309


In [12]:
print('*** In-Sample wheat data descriptive statistics ***')
print()
print('                   **** Farm-Level Metrics ****')
for ft in yes_no:
    print(f'The percentage of {ft} = {iswheat[ft].mean()*100:.2f}')
print()
print(np.round(iswheat[['farm_crop_yield','farm_land_m2']+contpm2].describe(),2))
print()
print('                   **** District Climate Metrics ****')
print(np.round(iswheat[climate].describe(),4))

*** In-Sample wheat data descriptive statistics ***

                   **** Farm-Level Metrics ****
The percentage of is_non_backward_caste = 28.53
The percentage of if_farmer_organisation = 2.94
The percentage of if_kisan_credit_card = 29.90
The percentage of if_soil_health_card = 1.85
The percentage of share_own_and_possessed = 85.94
The percentage of if_land_irrigated = 91.32
The percentage of primary_irrigation_canal = 8.34
The percentage of primary_irrigation_gw = 77.70
The percentage of primary_irrigation_mixed = 0.91
The percentage of primary_irrigation_msw = 2.77
The percentage of primary_irrigation_others = 1.65

       farm_crop_yield  farm_land_m2  cost_energy_pm2  cost_labour_pm2  \
count         10814.00      10814.00         10814.00         10814.00   
mean              0.45       7955.72             0.29             1.78   
std              15.21      11127.68             0.62            77.23   
min               0.00          0.40             0.00             0.00   

In [14]:
print('*** Out-of-Sample wheat data descriptive statistics ***')
print()
print('                   **** Farm-Level Metrics ****')
for ft in yes_no:
    print(f'The percentage of {ft} = {ooswheat[ft].mean()*100:.2f}')
print()
print(np.round(ooswheat[['farm_crop_yield','farm_land_m2']+contpm2].describe(),2))
print()
print('                   **** District Climate Metrics ****')
print(np.round(ooswheat[climate].describe(),4))

*** Out-of-Sample wheat data descriptive statistics ***

                   **** Farm-Level Metrics ****
The percentage of is_non_backward_caste = 27.96
The percentage of if_farmer_organisation = 2.96
The percentage of if_kisan_credit_card = 28.85
The percentage of if_soil_health_card = 1.63
The percentage of share_own_and_possessed = 85.89
The percentage of if_land_irrigated = 91.46
The percentage of primary_irrigation_canal = 7.91
The percentage of primary_irrigation_gw = 78.81
The percentage of primary_irrigation_mixed = 0.85
The percentage of primary_irrigation_msw = 2.59
The percentage of primary_irrigation_others = 1.52

       farm_crop_yield  farm_land_m2  cost_energy_pm2  cost_labour_pm2  \
count          2704.00       2704.00          2704.00          2704.00   
mean              0.31       7976.13             0.31             1.07   
std               0.14      11675.22             1.06             1.64   
min               0.00        161.87             0.00             0.0

In [49]:
print('Saving shared-district wheat Data')
iswheat.to_csv("C://Users/nb/Desktop/paper_2/xgboost_shap/feature_engineering/shared_insample_wheat.csv",index=False)
ooswheat.to_csv("C://Users/nb/Desktop/paper_2/xgboost_shap/feature_engineering/shared_oosample_wheat.csv",index=False)
print('wheat data saved ~~')

Saving shared-district wheat Data
wheat data saved ~~


In [4]:
iswheat=pd.read_csv(f"{drive}/xgboost_shap/feature_engineering/shared_insample_wheat.csv")
print('Number of districts in-sample wheat: ',iswheat.spei_district_code.nunique())
ooswheat=pd.read_csv(f"{drive}/xgboost_shap/feature_engineering/shared_oosample_wheat.csv")
print('OOS wheat',ooswheat.spei_district_code.nunique())

Number of districts in-sample wheat:  323
OOS wheat 309


### Mutually Exclusive (ME) District Split

In [40]:
ricedf['district_count']=ricedf[['nss_district_code19','farm_crop_yield']].copy().groupby('nss_district_code19').transform('count')
print(ricedf.shape)
print()
ricedf=ricedf[ricedf['district_count']>=5]
ricedf['district_ranks']=pd.qcut(ricedf['district_count'].copy(),duplicates='drop',labels=range(10),q=10)
rice_districts = (ricedf[['nss_district_code19', 'district_count', 'district_ranks']].drop_duplicates())
rice_districts

(22196, 94)



,nss_district_code19,district_count,district_ranks
3,106,9,0
12,107,28,1
46,119,7,0
55,121,26,1
81,122,18,0
...,...,...,...
22238,3625,13,0
22252,3627,20,0
22276,3629,16,0
22292,3630,14,0


In [41]:
train_rice_districts,test_rice_districts=train_test_split(rice_districts,test_size=0.2,stratify=rice_districts['district_ranks'],random_state=state_space)
isrice = ricedf[ricedf['nss_district_code19'].isin(train_rice_districts['nss_district_code19'])]
oosrice = ricedf[ricedf['nss_district_code19'].isin(test_rice_districts['nss_district_code19'])]
print(f'In-sample Data Size = {isrice.shape[0]} and Data Share = {isrice.shape[0]/ricedf.shape[0]:.2f}')
print(f'Out-of-sample Data Size = {oosrice.shape[0]} and Data Share = {oosrice.shape[0]/ricedf.shape[0]:.2f}')

In-sample Data Size = 17791 and Data Share = 0.80
Out-of-sample Data Size = 4405 and Data Share = 0.20


In [42]:
print('Verifying if districts are shared between test and train:')
print(len(set(isrice['nss_district_code19'])& set(oosrice['nss_district_code19'])))

Verifying if districts are shared between test and train:
0


In [43]:
print('*** In-Sample rice data descriptive statistics ***')
print()
print('                   **** Farm-Level Metrics ****')
for ft in yes_no:
    print(f'The percentage of {ft} = {isrice[ft].mean()*100:.2f}')
print()
print(np.round(isrice[['farm_crop_yield','farm_land_m2']+contpm2].describe(),2))
print()
print('                   **** District Climate Metrics ****')
print(np.round(isrice[climate].describe(),2))

*** In-Sample rice data descriptive statistics ***

                   **** Farm-Level Metrics ****
The percentage of is_non_backward_caste = 25.23
The percentage of if_farmer_organisation = 4.18
The percentage of if_kisan_credit_card = 17.82
The percentage of if_soil_health_card = 1.19
The percentage of share_own_and_possessed = 82.39
The percentage of if_land_irrigated = 62.89
The percentage of primary_irrigation_canal = 12.57
The percentage of primary_irrigation_gw = 45.15
The percentage of primary_irrigation_mixed = 1.02
The percentage of primary_irrigation_msw = 2.90
The percentage of primary_irrigation_others = 2.25

       farm_crop_yield  farm_land_m2  cost_energy_pm2  cost_labour_pm2  \
count         17791.00      17791.00         17791.00         17791.00   
mean              0.32       7814.71             0.12             1.30   
std               0.18       9518.26             0.36             1.76   
min               0.00         80.94             0.00             0.00   

In [44]:
print('*** Out-of-Sample rice data descriptive statistics ***')
print()
print('                   **** Farm-Level Metrics ****')
for ft in yes_no:
    print(f'The percentage of {ft} = {oosrice[ft].mean()*100:.2f}')
print()
print(np.round(oosrice[['farm_crop_yield','farm_land_m2']+contpm2].describe(),2))
print()
print('                   **** District Climate Metrics ****')
print(np.round(oosrice[climate].describe(),2))

*** Out-of-Sample rice data descriptive statistics ***

                   **** Farm-Level Metrics ****
The percentage of is_non_backward_caste = 29.90
The percentage of if_farmer_organisation = 4.15
The percentage of if_kisan_credit_card = 18.59
The percentage of if_soil_health_card = 1.50
The percentage of share_own_and_possessed = 82.20
The percentage of if_land_irrigated = 66.54
The percentage of primary_irrigation_canal = 16.12
The percentage of primary_irrigation_gw = 45.02
The percentage of primary_irrigation_mixed = 0.73
The percentage of primary_irrigation_msw = 3.63
The percentage of primary_irrigation_others = 2.32

       farm_crop_yield  farm_land_m2  cost_energy_pm2  cost_labour_pm2  \
count          4405.00       4405.00          4405.00          4405.00   
mean              0.34       8663.03             0.13             1.43   
std               0.18      13226.85             0.34             1.72   
min               0.00        121.41             0.00             0.0

In [45]:
print('Saving shared-district Rice Data')
isrice.to_csv(f"{drive}/xgboost_shap/feature_engineering/me_insample_rice.csv",index=False)
oosrice.to_csv(f"{drive}/xgboost_shap/feature_engineering/me_oosample_rice.csv",index=False)
print('Rice data saved ~~')

Saving shared-district Rice Data
Rice data saved ~~


In [21]:
### ---- Transformations ----
def column_transform(indf):
    warning_counts={}
    warn=[]
    indf=indf.copy()
    for col in indf.columns:
        #print(f"WORKING ON COLUMNS: {col}")
        if ("is" in col) or ("if" in col) or ("state" in col) or ('primary' in col):
        #    print('Dummy col: ', col)
            a=1
        else:
            with warnings.catch_warnings(record=True) as wlist:
                warnings.simplefilter("always")
           #     print('Yeo-Johnson on ', col)
                y = np.array(indf[col]).reshape(-1, 1)
                transformer = PowerTransformer(method='yeo-johnson')
                y_yeo = transformer.fit_transform(y)
                indf["yeo_" + col] = y_yeo.flatten()
    
                # Log transform only if all values are positive
                if np.all(indf[col] > 0):
           #         print(f'Log transform on {col}')
                    indf["ln_" + col] = np.log(indf[col])
                else:
                    a=1
           #         print(f"Skipping log transform on {col} due to non-positive values.")
            if wlist:
              warning_counts[col] = len(wlist)
            #  print(f"⚠️ Column '{col}' raised {len(wlist)} warning(s).")

    print("\n=== SUMMARY OF WARNINGS ===")
    for col, n in warning_counts.items():
        warn.append(col)
        print(f"{col}: {n} warnings")
            
    return [indf,warn]

### Transforming Data

#### Rice

#### In-Sample

In [22]:
isrice=pd.read_csv(f"{drive}/xgboost_shap/feature_engineering/shared_insample_rice.csv")
print(f'The shape of in-sample-rice data is {isrice.shape}')
isrice_x=column_transform(isrice)
isrice_x_df,warns=isrice_x[0],isrice_x[1]
print(' =============== ')

The shape of in-sample-rice data is (17756, 95)

=== SUMMARY OF WARNINGS ===
t2m_mean: 12 warnings
ssr_max: 5 warnings
ssr_min: 5 warnings


#### Rice

##### Out-of-Sample

In [23]:
oosrice=pd.read_csv(f"{drive}/xgboost_shap/feature_engineering/shared_oosample_rice.csv")
print(f'The shape of out-of-sample-rice data is {oosrice.shape}')
oosrice_x=column_transform(oosrice)
oosrice_x_df,warns=oosrice_x[0],oosrice_x[1]
print(' =============== ') ## there is problem with slhf minimum in the smaller sample so we drop this

The shape of out-of-sample-rice data is (4440, 95)

=== SUMMARY OF WARNINGS ===
t2m_mean: 13 warnings
ssr_max: 22 warnings
ssr_min: 22 warnings


In [24]:
## selecting the final columns for in-sample and out-of-sample based on warnings raised
finalcols=['yeo_farm_crop_yield', 'yeo_farm_land_m2', 'yeo_share_own_and_possessed',
           'yeo_cost_energy_pm2','yeo_cost_labour_pm2', 'yeo_cost_finance_pm2', 'yeo_cost_fert_pm2',
           'yeo_cost_pest_pm2', 'yeo_cost_machine_pm2', 'yeo_cost_irrigation_pm2', 'yeo_cost_seeds_pm2',
           'yeo_hh_cons_exp_pm_pm2',
           'yeo_median_speiSPEI-4MO', 'yeo_max_speiSPEI-4MO','yeo_min_speiSPEI-4MO', 'yeo_std_speiSPEI-4MO',
           'yeo_tp_mean','yeo_tp_std', 'ln_t2m_mean','yeo_t2m_max', 'yeo_t2m_min', 'yeo_t2m_std', 
           'yeo_slhf_mean', 'yeo_slhf_max', 'yeo_slhf_std', 'yeo_ssr_mean', 
           'ln_ssr_max', 'ln_ssr_min', 'yeo_ssr_std']
states=    ['state_10','state_11', 'state_12', 'state_13', 'state_14', 'state_15', 'state_16',
            'state_17', 'state_18', 'state_19', 'state_2', 'state_20', 'state_21',
            'state_22', 'state_23', 'state_24', 'state_25', 'state_26', 'state_27',
            'state_28', 'state_29', 'state_3', 'state_30', 'state_32', 'state_33',
            'state_34', 'state_36', 'state_4', 'state_5', 'state_6', 'state_7','state_8', 'state_9']
dummies=   [ 'if_land_irrigated', 'is_non_backward_caste','if_farmer_organisation', 
             'if_kisan_credit_card', 'if_soil_health_card','primary_irrigation_canal',
             'primary_irrigation_gw', 'primary_irrigation_mixed',
             'primary_irrigation_msw', 'primary_irrigation_others']

In [25]:
final_isrice=isrice_x_df[finalcols+dummies+states]
final_oosrice=oosrice_x_df[finalcols+dummies+states]
print(f'The shape of in-sample transformed rice data = {final_isrice.shape}')
print(f'The shape of out-of-sample transformed rice data = {final_oosrice.shape}')

The shape of in-sample transformed rice data = (17756, 72)
The shape of out-of-sample transformed rice data = (4440, 72)


In [26]:
print()
print('*** The In-Sample Data Descriptive Statistics ***')
for col in dummies:
    print(f'The percentage of {col} = {final_isrice[col].mean()*100:.2f}')
print()
np.round(final_isrice[finalcols].describe(),2)


*** The In-Sample Data Descriptive Statistics ***
The percentage of if_land_irrigated = 63.63
The percentage of is_non_backward_caste = 26.13
The percentage of if_farmer_organisation = 4.16
The percentage of if_kisan_credit_card = 18.12
The percentage of if_soil_health_card = 1.22
The percentage of primary_irrigation_canal = 13.16
The percentage of primary_irrigation_gw = 45.29
The percentage of primary_irrigation_mixed = 0.97
The percentage of primary_irrigation_msw = 3.02
The percentage of primary_irrigation_others = 2.27



,yeo_farm_crop_yield,yeo_farm_land_m2,yeo_share_own_and_possessed,yeo_cost_energy_pm2,yeo_cost_labour_pm2,yeo_cost_finance_pm2,yeo_cost_fert_pm2,yeo_cost_pest_pm2,yeo_cost_machine_pm2,yeo_cost_irrigation_pm2,...,yeo_t2m_max,yeo_t2m_min,yeo_t2m_std,yeo_slhf_mean,yeo_slhf_max,yeo_slhf_std,yeo_ssr_mean,ln_ssr_max,ln_ssr_min,yeo_ssr_std
count,17756.00,17756.00,17756.00,17756.00,17756.00,17756.00,17756.00,17756.00,17756.00,17756.00,...,17756.00,17756.00,17756.00,17756.00,17756.00,17756.00,17756.00,17756.00,17756.00,17756.00
mean,0.00,-0.00,-0.00,0.00,0.00,-0.00,0.00,0.00,-0.00,-0.00,...,0.00,-0.00,0.00,-0.00,-0.00,-0.00,0.00,13.69,11.40,0.00
std,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,...,1.00,1.00,1.00,1.00,1.00,1.00,1.00,0.05,0.33,1.00
min,-2.37,-4.18,-2.00,-0.58,-1.50,-0.40,-1.78,-1.00,-1.30,-0.65,...,-2.84,-5.00,-3.27,-5.96,-3.78,-3.37,-4.89,13.53,10.37,-3.24
25%,-0.69,-0.74,0.53,-0.58,-0.73,-0.40,-0.68,-1.00,-1.13,-0.65,...,-0.78,-0.45,-0.52,-0.56,-0.74,-0.49,-0.54,13.66,11.15,-0.78
50%,0.03,-0.02,0.53,-0.58,0.06,-0.40,0.05,-0.15,0.10,-0.65,...,0.04,-0.16,0.19,-0.01,0.06,0.15,0.06,13.68,11.39,0.02
75%,0.73,0.71,0.53,0.38,0.72,-0.40,0.66,0.79,0.76,1.06,...,0.70,0.36,0.61,0.60,0.70,0.51,0.51,13.72,11.65,0.70
max,4.23,4.79,0.53,2.08,3.54,2.71,3.00,2.06,2.88,1.89,...,2.89,3.62,10.10,3.55,4.52,5.97,6.24,13.90,12.54,3.27


In [27]:
print()
print('*** The Out-of-Sample Data Descriptive Statistics ***')
for col in dummies:
    print(f'The percentage of {col} = {final_oosrice[col].mean()*100:.2f}')
print()
np.round(final_oosrice[finalcols].describe(),2)


*** The Out-of-Sample Data Descriptive Statistics ***
The percentage of if_land_irrigated = 63.54
The percentage of is_non_backward_caste = 26.26
The percentage of if_farmer_organisation = 4.23
The percentage of if_kisan_credit_card = 17.36
The percentage of if_soil_health_card = 1.40
The percentage of primary_irrigation_canal = 13.72
The percentage of primary_irrigation_gw = 44.46
The percentage of primary_irrigation_mixed = 0.95
The percentage of primary_irrigation_msw = 3.13
The percentage of primary_irrigation_others = 2.25



,yeo_farm_crop_yield,yeo_farm_land_m2,yeo_share_own_and_possessed,yeo_cost_energy_pm2,yeo_cost_labour_pm2,yeo_cost_finance_pm2,yeo_cost_fert_pm2,yeo_cost_pest_pm2,yeo_cost_machine_pm2,yeo_cost_irrigation_pm2,...,yeo_t2m_max,yeo_t2m_min,yeo_t2m_std,yeo_slhf_mean,yeo_slhf_max,yeo_slhf_std,yeo_ssr_mean,ln_ssr_max,ln_ssr_min,yeo_ssr_std
count,4440.00,4440.00,4440.00,4440.00,4440.00,4440.00,4440.00,4440.00,4440.00,4440.00,...,4440.00,4440.00,4440.00,4440.00,4440.00,4440.00,4440.00,4440.00,4440.00,4440.00
mean,-0.00,0.00,-0.00,-0.00,-0.00,0.00,-0.00,0.00,-0.00,-0.00,...,0.00,0.00,0.00,0.00,0.00,-0.00,-0.00,13.69,11.40,-0.00
std,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,...,1.00,1.00,1.00,1.00,1.00,1.00,1.00,0.05,0.33,1.00
min,-2.37,-3.73,-1.95,-0.58,-1.51,-0.41,-1.79,-1.00,-1.31,-0.64,...,-2.86,-4.99,-3.18,-5.60,-3.64,-3.26,-4.71,13.53,10.37,-3.01
25%,-0.72,-0.75,0.55,-0.58,-0.75,-0.41,-0.70,-1.00,-1.12,-0.64,...,-0.78,-0.45,-0.53,-0.56,-0.71,-0.47,-0.55,13.66,11.17,-0.78
50%,0.00,0.01,0.55,-0.58,0.07,-0.41,0.05,-0.16,0.08,-0.64,...,0.02,-0.15,0.17,-0.02,0.06,0.13,0.02,13.69,11.39,0.01
75%,0.72,0.70,0.55,0.40,0.71,-0.41,0.69,0.81,0.74,1.01,...,0.70,0.36,0.60,0.60,0.69,0.50,0.47,13.73,11.63,0.73
max,3.96,4.16,0.55,2.08,3.34,2.65,3.04,2.06,2.88,1.91,...,2.86,3.43,10.59,3.57,4.83,5.61,6.11,13.88,12.54,3.28


In [28]:
final_isrice.to_csv(f"{drive}/xgboost_shap/feature_engineering/shared_yeoinsample_rice_vJan.csv",index=False)
print('Saving the final in-sample yeo-transformed for rice data ~') #the january version has information on types of primary irrigation

Saving the final in-sample yeo-transformed for rice data ~


In [36]:
final_isrice.columns

Index(['yeo_farm_crop_yield', 'yeo_farm_land_m2',
       'yeo_share_own_and_possessed', 'yeo_cost_energy_pm2',
       'yeo_cost_labour_pm2', 'yeo_cost_finance_pm2', 'yeo_cost_fert_pm2',
       'yeo_cost_pest_pm2', 'yeo_cost_machine_pm2', 'yeo_cost_irrigation_pm2',
       'yeo_cost_seeds_pm2', 'yeo_hh_cons_exp_pm_pm2',
       'yeo_median_speiSPEI-4MO', 'yeo_max_speiSPEI-4MO',
       'yeo_min_speiSPEI-4MO', 'yeo_std_speiSPEI-4MO', 'yeo_tp_mean',
       'yeo_tp_std', 'ln_t2m_mean', 'yeo_t2m_max', 'yeo_t2m_min',
       'yeo_t2m_std', 'yeo_slhf_mean', 'yeo_slhf_max', 'yeo_slhf_std',
       'yeo_ssr_mean', 'ln_ssr_max', 'ln_ssr_min', 'yeo_ssr_std',
       'if_land_irrigated', 'is_non_backward_caste', 'if_farmer_organisation',
       'if_kisan_credit_card', 'if_soil_health_card',
       'primary_irrigation_canal', 'primary_irrigation_gw',
       'primary_irrigation_mixed', 'primary_irrigation_msw',
       'primary_irrigation_others', 'state_10', 'state_11', 'state_12',
       'state_13', 'sta

In [29]:
final_oosrice.to_csv(f"{drive}/xgboost_shap/feature_engineering/shared_yeooosample_rice_vJan.csv",index=False)
print('Saving the final out-of-sample yeo-transformed for rice data ~') #the january version has information on types of primary irrigation

Saving the final out-of-sample yeo-transformed for rice data ~


### Transforming Data

#### Wheat

#### In-Sample

In [50]:
iswheat=pd.read_csv(f"{drive}/xgboost_shap/feature_engineering/shared_insample_wheat.csv")
print(f'The shape of in-sample-wheat data is {iswheat.shape}')
iswheat_x=column_transform(iswheat)
iswheat_x_df,warns=iswheat_x[0],iswheat_x[1]
print(' =============== ')

The shape of in-sample-wheat data is (10814, 95)

=== SUMMARY OF WARNINGS ===
t2m_mean: 11 warnings
t2m_max: 8 warnings
ssr_std: 7 warnings


#### Wheat

##### Out-of-Sample

In [52]:
ooswheat=pd.read_csv(f"{drive}/xgboost_shap/feature_engineering/shared_oosample_wheat.csv")
print(f'The shape of out-of-sample-wheat data is {ooswheat.shape}')
ooswheat_x=column_transform(ooswheat)
ooswheat_x_df,warns=ooswheat_x[0],ooswheat_x[1]
print(' =============== ') ## there is problem with slhf minimum in the smaller sample so we drop this

The shape of out-of-sample-wheat data is (2704, 95)

=== SUMMARY OF WARNINGS ===
t2m_mean: 10 warnings
t2m_max: 10 warnings
ssr_std: 8 warnings


In [53]:
## selecting the final columns for in-sample and out-of-sample based on warnings raised
finalcols=['yeo_farm_crop_yield', 'yeo_farm_land_m2', 'yeo_share_own_and_possessed',
           'yeo_cost_energy_pm2','yeo_cost_labour_pm2', 'yeo_cost_finance_pm2', 'yeo_cost_fert_pm2',
           'yeo_cost_pest_pm2', 'yeo_cost_machine_pm2', 'yeo_cost_irrigation_pm2', 'yeo_cost_seeds_pm2',
           'yeo_hh_cons_exp_pm_pm2',
           'yeo_median_speiSPEI-4MO', 'yeo_max_speiSPEI-4MO','yeo_min_speiSPEI-4MO', 'yeo_std_speiSPEI-4MO',
           'yeo_tp_mean','yeo_tp_std', 'ln_t2m_mean','yeo_t2m_max', 'yeo_t2m_min', 'yeo_t2m_std', 
           'yeo_slhf_mean', 'yeo_slhf_max', 'yeo_slhf_std', 'yeo_ssr_mean', 
           'ln_ssr_max', 'ln_ssr_min', 'yeo_ssr_std']
states=    ['state_10','state_11', 'state_12', 'state_13', 'state_14', 'state_15', 'state_16',
            'state_17', 'state_18', 'state_19', 'state_2', 'state_20', 'state_21',
            'state_22', 'state_23', 'state_24', 'state_25', 'state_26', 'state_27',
            'state_28', 'state_29', 'state_3', 'state_30', 'state_32', 'state_33',
            'state_34', 'state_36', 'state_4', 'state_5', 'state_6', 'state_7','state_8', 'state_9']
dummies=   [ 'if_land_irrigated', 'is_non_backward_caste','if_farmer_organisation', 
             'if_kisan_credit_card', 'if_soil_health_card','primary_irrigation_canal',
             'primary_irrigation_gw', 'primary_irrigation_mixed',
             'primary_irrigation_msw', 'primary_irrigation_others']

In [54]:
final_iswheat=iswheat_x_df[finalcols+dummies+states]
final_ooswheat=ooswheat_x_df[finalcols+dummies+states]
print(f'The shape of in-sample transformed wheat data = {final_iswheat.shape}')
print(f'The shape of out-of-sample transformed wheat data = {final_ooswheat.shape}')

The shape of in-sample transformed wheat data = (10814, 72)
The shape of out-of-sample transformed wheat data = (2704, 72)


In [55]:
print()
print('*** The In-Sample Data Descriptive Statistics ***')
for col in dummies:
    print(f'The percentage of {col} = {final_iswheat[col].mean()*100:.2f}')
print()
np.round(final_iswheat[finalcols].describe(),2)


*** The In-Sample Data Descriptive Statistics ***
The percentage of if_land_irrigated = 91.32
The percentage of is_non_backward_caste = 28.53
The percentage of if_farmer_organisation = 2.94
The percentage of if_kisan_credit_card = 29.90
The percentage of if_soil_health_card = 1.85
The percentage of primary_irrigation_canal = 8.34
The percentage of primary_irrigation_gw = 77.70
The percentage of primary_irrigation_mixed = 0.91
The percentage of primary_irrigation_msw = 2.77
The percentage of primary_irrigation_others = 1.65



,yeo_farm_crop_yield,yeo_farm_land_m2,yeo_share_own_and_possessed,yeo_cost_energy_pm2,yeo_cost_labour_pm2,yeo_cost_finance_pm2,yeo_cost_fert_pm2,yeo_cost_pest_pm2,yeo_cost_machine_pm2,yeo_cost_irrigation_pm2,...,yeo_t2m_max,yeo_t2m_min,yeo_t2m_std,yeo_slhf_mean,yeo_slhf_max,yeo_slhf_std,yeo_ssr_mean,ln_ssr_max,ln_ssr_min,yeo_ssr_std
count,10814.00,10814.00,10814.00,10814.00,10814.00,10814.00,10814.00,10814.00,10814.00,10814.00,...,10814.00,10814.00,10814.00,10814.00,10814.00,10814.00,10814.00,10814.00,10814.00,10814.0
mean,0.00,-0.00,0.00,0.00,0.00,-0.00,-0.00,0.00,-0.00,-0.00,...,-0.00,0.00,0.00,0.00,-0.00,0.00,-0.00,13.85,12.38,0.0
std,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,...,1.00,1.00,1.00,1.00,1.00,1.00,1.00,0.04,0.43,0.0
min,-2.80,-8.39,-2.20,-0.85,-1.42,-0.32,-2.33,-1.01,-1.66,-0.89,...,-2.94,-3.05,-5.68,-2.82,-2.40,-4.55,-3.65,13.60,10.35,0.0
25%,-0.58,-0.73,0.47,-0.85,-0.79,-0.32,-0.61,-1.01,-0.66,-0.89,...,-0.52,-0.62,-0.69,-0.72,-0.79,-0.54,-0.49,13.83,12.23,0.0
50%,0.05,-0.01,0.47,-0.85,0.07,-0.32,-0.00,-0.11,0.04,-0.75,...,0.06,0.04,-0.15,-0.04,-0.05,-0.01,0.00,13.85,12.47,0.0
75%,0.67,0.70,0.47,0.95,0.71,-0.32,0.63,0.78,0.68,1.02,...,0.65,0.67,0.77,0.74,0.71,0.61,0.38,13.87,12.65,0.0
max,8.73,4.20,0.47,1.94,3.40,3.27,2.86,2.11,3.46,1.99,...,2.57,7.55,2.05,2.59,2.70,2.92,3.99,13.98,13.04,0.0


In [56]:
print()
print('*** The Out-of-Sample Data Descriptive Statistics ***')
for col in dummies:
    print(f'The percentage of {col} = {final_ooswheat[col].mean()*100:.2f}')
print()
np.round(final_ooswheat[finalcols].describe(),2)


*** The Out-of-Sample Data Descriptive Statistics ***
The percentage of if_land_irrigated = 91.46
The percentage of is_non_backward_caste = 27.96
The percentage of if_farmer_organisation = 2.96
The percentage of if_kisan_credit_card = 28.85
The percentage of if_soil_health_card = 1.63
The percentage of primary_irrigation_canal = 7.91
The percentage of primary_irrigation_gw = 78.81
The percentage of primary_irrigation_mixed = 0.85
The percentage of primary_irrigation_msw = 2.59
The percentage of primary_irrigation_others = 1.52



,yeo_farm_crop_yield,yeo_farm_land_m2,yeo_share_own_and_possessed,yeo_cost_energy_pm2,yeo_cost_labour_pm2,yeo_cost_finance_pm2,yeo_cost_fert_pm2,yeo_cost_pest_pm2,yeo_cost_machine_pm2,yeo_cost_irrigation_pm2,...,yeo_t2m_max,yeo_t2m_min,yeo_t2m_std,yeo_slhf_mean,yeo_slhf_max,yeo_slhf_std,yeo_ssr_mean,ln_ssr_max,ln_ssr_min,yeo_ssr_std
count,2704.00,2704.00,2704.00,2704.00,2704.00,2704.00,2704.00,2704.00,2704.00,2704.00,...,2704.00,2704.00,2704.00,2704.00,2704.00,2704.00,2704.00,2704.00,2704.00,2704.0
mean,-0.00,-0.00,0.00,-0.00,0.00,-0.00,0.00,-0.00,-0.00,0.00,...,0.00,0.00,-0.00,-0.00,-0.00,0.00,-0.00,13.85,12.38,0.0
std,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,...,1.00,1.00,1.00,1.00,1.00,1.00,1.00,0.04,0.42,0.0
min,-2.71,-3.74,-2.19,-0.85,-1.43,-0.31,-2.38,-1.03,-1.71,-0.90,...,-2.99,-3.00,-5.64,-2.76,-2.44,-3.07,-3.76,13.64,10.35,0.0
25%,-0.57,-0.73,0.48,-0.85,-0.76,-0.31,-0.61,-1.03,-0.62,-0.90,...,-0.54,-0.65,-0.71,-0.70,-0.80,-0.53,-0.47,13.83,12.24,0.0
50%,0.01,-0.02,0.48,-0.85,0.08,-0.31,-0.01,-0.10,0.04,-0.74,...,0.06,-0.03,-0.14,-0.02,-0.14,0.00,-0.02,13.85,12.47,0.0
75%,0.70,0.73,0.48,0.94,0.70,-0.31,0.59,0.78,0.66,0.99,...,0.67,0.67,0.76,0.73,0.74,0.61,0.35,13.87,12.65,0.0
max,4.73,4.33,0.48,1.95,3.13,3.36,2.77,2.15,2.93,2.00,...,2.63,5.19,2.08,2.59,2.58,2.95,3.83,13.98,13.04,0.0


In [57]:
final_iswheat.to_csv(f"{drive}/xgboost_shap/feature_engineering/shared_yeoinsample_wheat_vJan.csv",index=False)
print('Saving the final in-sample yeo-transformed for wheat data ~') #the january version has information on types of primary irrigation

Saving the final in-sample yeo-transformed for wheat data ~


In [58]:
final_ooswheat.to_csv(f"{drive}xgboost_shap/feature_engineering/shared_yeooosample_wheat_vJan.csv",index=False)
print('Saving the final out-of-sample yeo-transformed for wheat data ~') #the january version has information on types of primary irrigation

Saving the final out-of-sample yeo-transformed for wheat data ~


### Preparing MAP DATA

In [2]:
oosrice=pd.read_csv(f"{drive}/xgboost_shap/feature_engineering/shared_oosample_rice.csv")
oosrice_districts,oosrice_t2m_mean=oosrice['nss_district_code19'].nunique(),oosrice['t2m_mean'].nunique()
print(f'Number of rice OOS districts = {oosrice_districts}')
print(f'Number of rice unique t2m_mean = {oosrice_t2m_mean}')
oosrice_district_codes=oosrice[['nss_district_code19','t2m_mean']].copy().drop_duplicates().reset_index(drop=True).sort_values(by='t2m_mean').reset_index()
oosrice_district_codes=oosrice_district_codes.rename(columns={'index':'t2m_mean_rank'})
print(f'The match of t2m to district codes = {oosrice_district_codes.shape}')
print(oosrice_district_codes.describe())

Number of rice OOS districts = 474
Number of rice unique t2m_mean = 474
The match of t2m to district codes = (474, 3)
       t2m_mean_rank  nss_district_code19    t2m_mean
count     474.000000           474.000000  474.000000
mean      236.500000          1748.381857  297.041307
std       136.976275           939.882193    3.760567
min         0.000000           102.000000  273.951228
25%       118.250000           961.250000  296.849615
50%       236.500000          1820.500000  297.939634
75%       354.750000          2329.500000  298.688922
max       473.000000          3631.000000  301.968284


In [16]:
mapfile=pd.read_csv(f"{drive}/nss_to_spei_code.csv")
oosricemap=pd.merge(oosrice_district_codes,
                   mapfile,on='nss_district_code19')
print(oosricemap.shape)
oosricemap.to_csv(f"{drive}/xgboost_shap/feature_engineering/oosrice_district_map.csv",index=False)

(474, 4)


In [17]:
oosricemap

,t2m_mean_rank,nss_district_code19,t2m_mean,spei_district_code
0,468,109,273.951228,413
1,69,501,275.359241,607
2,436,1101,275.495355,271
3,71,502,276.238756,468
4,360,201,278.732244,82
...,...,...,...,...
469,228,3319,301.732842,692
470,26,3306,301.770715,702
471,158,3303,301.821819,676
472,306,3315,301.825898,668


In [3]:
ooswheat=pd.read_csv(f"{drive}/xgboost_shap/feature_engineering/shared_oosample_wheat.csv")
ooswheat_districts,ooswheat_t2m_mean=ooswheat['nss_district_code19'].nunique(),ooswheat['t2m_mean'].nunique()
print(f'Number of wheat OOS districts = {ooswheat_districts}')
print(f'Number of wheat unique t2m_mean = {ooswheat_t2m_mean}')
ooswheat_district_codes=ooswheat[['nss_district_code19','t2m_mean']].copy().drop_duplicates().reset_index(drop=True).sort_values(by='t2m_mean').reset_index()
ooswheat_district_codes=ooswheat_district_codes.rename(columns={'index':'t2m_mean_rank'})
print(f'The match of t2m to district codes = {ooswheat_district_codes.shape}')
print(ooswheat_district_codes.describe())

Number of wheat OOS districts = 309
Number of wheat unique t2m_mean = 309
The match of t2m to district codes = (309, 3)
       t2m_mean_rank  nss_district_code19    t2m_mean
count     309.000000           309.000000  309.000000
mean      154.000000          1242.378641  297.533011
std        89.344838           779.064713    4.750332
min         0.000000           105.000000  267.757624
25%        77.000000           806.000000  297.487533
50%       154.000000           951.000000  298.632737
75%       231.000000          2223.000000  299.575082
max       308.000000          2731.000000  303.247717


In [4]:
mapfile=pd.read_csv(f"{drive}/nss_to_spei_code.csv")
ooswheatmap=pd.merge(ooswheat_district_codes,
                   mapfile,on='nss_district_code19')
print(ooswheatmap.shape)
ooswheatmap.to_csv(f"{drive}/xgboost_shap/feature_engineering/ooswheat_district_map.csv",index=False)

(309, 4)


In [5]:
ooswheatmap

,t2m_mean_rank,nss_district_code19,t2m_mean,spei_district_code
0,226,118,267.757624,425
1,169,501,270.114124,607
2,194,502,271.094948,468
3,301,204,272.782201,229
4,216,201,274.218716,82
...,...,...,...,...
304,306,2729,301.889386,554
305,289,2716,302.078993,492
306,300,2705,302.407792,443
307,92,2715,302.817714,547


### Transforming the Mutually-Exclusive District Data

In [13]:
isrice=pd.read_csv(f"{drive}/xgboost_shap/feature_engineering/me_insample_rice.csv")
print(f'The shape of in-sample-rice data is {isrice.shape}')
isrice_x=column_transform(isrice)
isrice_x_df,warns=isrice_x[0],isrice_x[1]
print(' =============== ')

The shape of in-sample-rice data is (17791, 93)

=== SUMMARY OF WARNINGS ===
t2m_mean: 11 warnings
ssr_max: 7 warnings
ssr_min: 7 warnings


#### Rice

##### Out-of-Sample

In [14]:
oosrice=pd.read_csv(f"{drive}/xgboost_shap/feature_engineering/me_oosample_rice.csv")
print(f'The shape of out-of-sample-rice data is {oosrice.shape}')
oosrice_x=column_transform(oosrice)
oosrice_x_df,warns=oosrice_x[0],oosrice_x[1]
print(' =============== ') ## there is problem with slhf minimum in the smaller sample so we drop this

The shape of out-of-sample-rice data is (4405, 93)

=== SUMMARY OF WARNINGS ===
t2m_mean: 9 warnings
t2m_max: 1 warnings
slhf_max: 12 warnings
slhf_min: 21 warnings
ssr_mean: 19 warnings
ssr_max: 19 warnings
ssr_min: 10 warnings


In [19]:
## selecting the final columns for in-sample and out-of-sample based on warnings raised
finalcols=['yeo_farm_crop_yield', 'yeo_farm_land_m2', 'yeo_share_own_and_possessed',
            'yeo_cost_energy_pm2','yeo_cost_labour_pm2', 'yeo_cost_finance_pm2', 'yeo_cost_pest_pm2',
            'yeo_cost_machine_pm2', 'yeo_cost_irrigation_pm2', 'yeo_cost_seeds_pm2','yeo_hh_cons_exp_pm_pm2',
            'yeo_median_speiSPEI-4MO', 'yeo_max_speiSPEI-4MO','yeo_min_speiSPEI-4MO', 'yeo_std_speiSPEI-4MO',
            'yeo_tp_mean','yeo_tp_std', 'ln_t2m_mean','yeo_t2m_max', 'yeo_t2m_min', 'yeo_t2m_std', 
            'yeo_slhf_mean', 'yeo_slhf_max', 'yeo_slhf_std', 'yeo_ssr_mean', 
            'ln_ssr_max', 'ln_ssr_min', 'yeo_ssr_std']
states=    ['state_10','state_11', 'state_12', 'state_13', 'state_14', 'state_15', 'state_16',
            'state_17', 'state_18', 'state_19', 'state_2', 'state_20', 'state_21',
            'state_22', 'state_23', 'state_24', 'state_25', 'state_26', 'state_27',
            'state_28', 'state_29', 'state_3', 'state_30', 'state_32', 'state_33',
            'state_34', 'state_36', 'state_4', 'state_5', 'state_6', 'state_7','state_8', 'state_9']
dummies=   [ 'if_land_irrigated', 'is_non_backward_caste','if_farmer_organisation', 
            'if_kisan_credit_card', 'if_soil_health_card']

In [20]:
final_isrice=isrice_x_df[finalcols+dummies+states]
final_oosrice=oosrice_x_df[finalcols+dummies+states]
print(f'The shape of in-sample transformed rice data = {final_isrice.shape}')
print(f'The shape of out-of-sample transformed rice data = {final_oosrice.shape}')

The shape of in-sample transformed rice data = (17791, 66)
The shape of out-of-sample transformed rice data = (4405, 66)


In [21]:
print()
print('*** The In-Sample Data Descriptive Statistics ***')
for col in dummies:
    print(f'The percentage of {col} = {final_isrice[col].mean()*100:.2f}')
print()
np.round(final_isrice[finalcols].describe(),2)


*** The In-Sample Data Descriptive Statistics ***
The percentage of if_land_irrigated = 62.89
The percentage of is_non_backward_caste = 25.23
The percentage of if_farmer_organisation = 4.18
The percentage of if_kisan_credit_card = 17.82
The percentage of if_soil_health_card = 1.19



,yeo_farm_crop_yield,yeo_farm_land_m2,yeo_share_own_and_possessed,yeo_cost_energy_pm2,yeo_cost_labour_pm2,yeo_cost_finance_pm2,yeo_cost_pest_pm2,yeo_cost_machine_pm2,yeo_cost_irrigation_pm2,yeo_cost_seeds_pm2,...,yeo_t2m_max,yeo_t2m_min,yeo_t2m_std,yeo_slhf_mean,yeo_slhf_max,yeo_slhf_std,yeo_ssr_mean,ln_ssr_max,ln_ssr_min,yeo_ssr_std
count,17791.00,17791.00,17791.00,17791.00,17791.00,17791.00,17791.00,17791.00,17791.00,17791.00,...,17791.00,17791.00,17791.00,17791.00,17791.00,17791.00,17791.00,17791.00,17791.00,17791.00
mean,-0.00,0.00,-0.00,0.00,0.00,-0.00,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,-0.00,0.00,13.69,11.41,0.00
std,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,...,1.00,1.00,1.00,1.00,1.00,1.00,1.00,0.05,0.33,1.00
min,-2.39,-4.04,-1.99,-0.57,-1.48,-0.40,-0.99,-1.29,-0.66,-1.33,...,-2.79,-4.67,-3.07,-3.19,-3.65,-3.50,-4.35,13.53,10.46,-2.76
25%,-0.68,-0.75,0.53,-0.57,-0.76,-0.40,-0.99,-1.16,-0.66,-1.33,...,-0.76,-0.44,-0.51,-0.47,-0.75,-0.50,-0.52,13.66,11.16,-0.75
50%,0.03,-0.01,0.53,-0.57,0.07,-0.40,-0.15,0.09,-0.66,0.08,...,0.05,-0.20,0.21,-0.03,0.04,0.15,0.04,13.68,11.40,0.02
75%,0.72,0.72,0.53,0.24,0.72,-0.40,0.78,0.76,1.09,0.73,...,0.71,0.37,0.60,0.55,0.75,0.53,0.50,13.73,11.66,0.69
max,4.24,4.70,0.53,2.09,3.52,2.73,2.06,2.85,1.88,2.77,...,2.87,3.54,9.36,3.46,4.48,5.69,5.03,13.90,12.46,3.17


In [22]:
print()
print('*** The Out-of-Sample Data Descriptive Statistics ***')
for col in dummies:
    print(f'The percentage of {col} = {final_oosrice[col].mean()*100:.2f}')
print()
np.round(final_oosrice[finalcols].describe(),2)


*** The Out-of-Sample Data Descriptive Statistics ***
The percentage of if_land_irrigated = 66.54
The percentage of is_non_backward_caste = 29.90
The percentage of if_farmer_organisation = 4.15
The percentage of if_kisan_credit_card = 18.59
The percentage of if_soil_health_card = 1.50



,yeo_farm_crop_yield,yeo_farm_land_m2,yeo_share_own_and_possessed,yeo_cost_energy_pm2,yeo_cost_labour_pm2,yeo_cost_finance_pm2,yeo_cost_pest_pm2,yeo_cost_machine_pm2,yeo_cost_irrigation_pm2,yeo_cost_seeds_pm2,...,yeo_t2m_max,yeo_t2m_min,yeo_t2m_std,yeo_slhf_mean,yeo_slhf_max,yeo_slhf_std,yeo_ssr_mean,ln_ssr_max,ln_ssr_min,yeo_ssr_std
count,4405.00,4405.00,4405.00,4405.00,4405.00,4405.00,4405.00,4405.00,4405.00,4405.00,...,4405.00,4405.00,4405.00,4405.00,4405.0,4405.00,4405.0,4405.00,4405.00,4405.00
mean,-0.00,-0.00,-0.00,0.00,-0.00,-0.00,-0.00,-0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.0,-0.00,0.0,13.69,11.35,0.00
std,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,...,1.00,1.00,1.00,1.00,0.0,1.00,0.0,0.05,0.33,1.00
min,-2.32,-4.18,-1.99,-0.61,-1.59,-0.43,-1.03,-1.37,-0.61,-1.28,...,-2.49,-6.01,-2.84,-5.02,0.0,-2.81,0.0,13.58,10.37,-2.75
25%,-0.71,-0.72,0.54,-0.61,-0.66,-0.43,-1.03,-1.02,-0.61,-1.28,...,-0.64,-0.48,-0.47,-0.75,0.0,-0.49,0.0,13.65,11.14,-0.79
50%,0.01,-0.00,0.54,-0.61,0.03,-0.43,-0.10,0.06,-0.61,0.07,...,0.03,-0.09,0.04,-0.01,0.0,0.08,0.0,13.69,11.36,0.10
75%,0.74,0.71,0.54,0.75,0.71,-0.43,0.76,0.73,0.87,0.71,...,0.81,0.36,0.59,0.68,0.0,0.45,0.0,13.71,11.54,0.78
max,4.28,4.13,0.54,2.02,3.22,2.57,2.07,3.04,1.96,2.46,...,2.30,3.38,4.81,2.82,0.0,3.92,0.0,13.86,12.54,2.92


In [ ]:
print(final_isrice.columns)
print(final_oosrice.columns)

In [23]:
final_isrice.to_csv(f"{drive}/xgboost_shap/feature_engineering/me_yeoinsample_rice.csv",index=False)
print('Saving the final in-sample yeo-transformed for rice data ~')

Saving the final in-sample yeo-transformed for rice data ~


In [24]:
final_oosrice.to_csv("C://Users/nb/Desktop/paper_2/xgboost_shap/feature_engineering/me_yeooosample_rice.csv",index=False)
print('Saving the final out-of-sample yeo-transformed for rice data ~')

Saving the final out-of-sample yeo-transformed for rice data ~


##### Wheat

In [ ]:
iswheat=pd.read_csv(f"{drive}/xgboost_shap/feature_engineering/insample_wheat.csv")
print(f'The shape of in-sample-wheat data is {iswheat.shape}')
iswheat_x=column_transform(iswheat)
iswheat_x_df,warns=iswheat_x[0],iswheat_x[1]
print(' =============== ')
#keeping yeo-transformations where successful, and keeping log where yeo did not work
finalcols=[]
for col in warns:
    if iswheat_x_df[f'yeo_{col}'].min() == iswheat_x_df[f'yeo_{col}'].max():
        use_col=f'ln_{col}'
    elif iswheat_x_df[f'ln_{col}'].min() == iswheat_x_df[f'ln_{col}'].max():
        use_col=f'yeo_{col}'
    elif iswheat_x_df[f'yeo_{col}'].std() > iswheat_x_df[f'yeo_{col}'].std():
        use_col=f'ln_{col}'
    else:
        use_col=f'ln_{col}'
    finalcols.append(use_col)
print()
print(finalcols)
#
#iswheat_x.to_csv("C://Users/nb/Desktop/paper_2/istransformedwheatdf.csv",index=False)

In [ ]:
finalcols=['yeo_farm_crop_yield', 'yeo_farm_land_m2', 'yeo_share_own_and_possessed',
            'yeo_cost_energy_pm2','yeo_cost_labour_pm2', 'yeo_cost_finance_pm2', 'yeo_cost_pest_pm2',
            'yeo_cost_machine_pm2', 'yeo_cost_irrigation_pm2', 'yeo_cost_seeds_pm2','yeo_hh_cons_exp_pm_pm2',
            'yeo_median_speiSPEI-4MO', 'yeo_max_speiSPEI-4MO','yeo_min_speiSPEI-4MO', 'yeo_std_speiSPEI-4MO',
            'yeo_tp_mean','yeo_tp_std', 'ln_t2m_mean','ln_t2m_max', 'yeo_t2m_min', 'yeo_t2m_std', 
            'yeo_slhf_mean', 'yeo_slhf_max', 'yeo_slhf_min','yeo_slhf_std', 'ln_ssr_mean', 
            'ln_ssr_max', 'ln_ssr_min', 'ln_ssr_std']
print()
for col in dummies:
    print(f'The percentage of {col} = {np.round(iswheat_x_df[col].mean()*100,2)}')
print()
final_iswheat=iswheat_x_df[finalcols+dummies+states]
print(f'The shape of in-sample transformed wheat data = {final_iswheat.shape}')
np.round(iswheat_x_df[finalcols].describe(),2)

In [ ]:
final_iswheat.to_csv(f"{drive}/xgboost_shap/feature_engineering/yeoinsample_wheat.csv")
print('Saving the final in-sample yeo-transformed for wheat data ~')

#### Transforming out-of-sample data
The idea here is to maintain parity in the columns between in-sample and out-of-sample data so that predictions make sense.

##### Rice

In [ ]:
ooswheat=pd.read_csv(f"{drive}/xgboost_shap/feature_engineering/oosample_wheat.csv")
print(f'The shape of out-of-sample-wheat data is {ooswheat.shape}')
ooswheat_x=column_transform(ooswheat)
ooswheat_x_df,warns=ooswheat_x[0],ooswheat_x[1]
print(' =============== ')

In [ ]:
finalcols=['yeo_farm_crop_yield', 'yeo_farm_land_m2', 'yeo_share_own_and_possessed',
            'yeo_cost_energy_pm2','yeo_cost_labour_pm2', 'yeo_cost_finance_pm2', 'yeo_cost_pest_pm2',
            'yeo_cost_machine_pm2', 'yeo_cost_irrigation_pm2', 'yeo_cost_seeds_pm2','yeo_hh_cons_exp_pm_pm2',
            'yeo_median_speiSPEI-4MO', 'yeo_max_speiSPEI-4MO','yeo_min_speiSPEI-4MO', 'yeo_std_speiSPEI-4MO',
            'yeo_tp_mean','yeo_tp_std', 'ln_t2m_mean','ln_t2m_max', 'yeo_t2m_min', 'yeo_t2m_std', 
            'yeo_slhf_mean', 'yeo_slhf_max', 'yeo_slhf_min','yeo_slhf_std', 'ln_ssr_mean', 
            'ln_ssr_max', 'ln_ssr_min', 'ln_ssr_std']
final_ooswheat=ooswheat_x_df[finalcols+dummies+states]
print()
for col in dummies:
    print(f'The percentage of {col} = {np.round(final_ooswheat[col].mean()*100,2)}')
print()
print(f'The shape of out-of-sample transformed wheat data = {final_ooswheat.shape}')
np.round(final_ooswheat[finalcols].describe(),2)

In [ ]:
final_ooswheat.to_csv(f"{drive}/xgboost_shap/feature_engineering/yeooosample_wheat.csv")
print('Saving the final out-of-sample yeo-transformed for wheat data ~')